# Trial-by-Trial Updating of Prior Beliefs in the IBL Decision-Making Task

## Research Question

How is trial-by-trial prior information represented in neural activity
during decision-making?

## Main Hypothesis

Prior information accumulated from previous trials is represented in
pre-stimulus neural activity and varies across brain regions and
task-relevant temporal epochs.

## Analysis Strategy

We first establish whether current choice can be decoded from neural
activity. We then examine whether decoding depends on previous-trial
outcome and when choice-related information emerges relative to stimulus
onset.

The final analysis tests whether neural activity directly represents
a trial-by-trial estimate of subjective prior belief.

In [2]:

# Analysis setup

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score,
)

RANDOM_STATE = 42
N_SPLITS = 3
MIN_UNITS = 5
MIN_TRIALS = 40
MIN_CONDITION_TRIALS = 30

In [3]:

# Load project data

data = download_data("stimOn")
meta = load_metadata("stimOn")

all_pids = (
    meta.clusters["pid"]
    .dropna()
    .unique()
)

print(
    f"Number of probe insertions: {len(all_pids)}"
)

NameError: name 'download_data' is not defined

In [ ]:
BROAD_TARGET_REGIONS = {
    "Sensory": [
        "VISp", "VISl", "VISam",
        "VISpm", "VISrl", "VISa"
    ],

    "Motor": [
        "MOs", "MOp"
    ],

    "Memory": [
        "CA1", "CA2", "CA3",
        "DG", "SUB"
    ],

    "Integrative": [
        "PL", "ILA", "ORBpt",
        "ORBvl", "ORBm", "PFC",
        "ACAd", "ACAv"
    ],
}

## 5. Baseline Choice Decoding

Before testing trial-by-trial prior representations, we first establish
whether current choice can be decoded from pre-stimulus neural activity.

For each probe insertion and brain-region group, neural activity is averaged
over the late pre-stimulus window (-0.4 to 0 s). A binary logistic regression
classifier is then trained to distinguish left versus right choices using
cross-validation.

This analysis serves as a baseline for the subsequent history-dependent
and subjective-prior decoding analyses.

In [ ]:

# Baseline choice decoding


BASELINE_WINDOW = (-0.4, 0.0)


def prepare_choice_decoder_data(
    pid,
    region_acronyms,
    meta,
    time_window=BASELINE_WINDOW,
):
    """
    Prepare neural features and current-choice labels
    for baseline decoding.

    Parameters
    ----------
    pid : str
        Probe insertion identifier.

    region_acronyms : list
        Brain-region acronyms included in the analysis.

    meta : object
        IBL metadata object.

    time_window : tuple
        Start and end time of the neural analysis window.

    Returns
    -------
    X : np.ndarray or None
        Trial-by-unit neural activity matrix.

    y : np.ndarray or None
        Binary current-choice labels.
        Left choice = 1
        Right choice = 0
    """

    # --------------------------------------------------------
    # Select clusters from the requested probe insertion
    # and brain regions.


    clusters = meta.clusters[
        (meta.clusters["pid"] == pid)
        & (
            meta.clusters["acronym"]
            .isin(region_acronyms)
        )
    ]

    if len(clusters) < MIN_UNITS:
        return None, None

    uuids = clusters["uuids"].tolist()

    # --------------------------------------------------------
    # Load PSTH data


    psth, _, trials = get_psth_for_clusters(
        uuids,
        meta,
    )

    if isinstance(psth, list):
        psth = psth[0]

    if isinstance(trials, list):
        trials = trials[0]

    if psth.shape[1] < MIN_UNITS:
        return None, None

    # --------------------------------------------------------
    # Select the requested temporal window


    start_time, end_time = time_window

    time_mask = (
        (meta.times >= start_time)
        & (meta.times < end_time)
    )

    if not np.any(time_mask):
        return None, None

    # Average firing activity within the selected window.
    X = psth[
        :, :, time_mask
    ].mean(axis=2)

    # --------------------------------------------------------
    # Select valid choice trials


    valid_mask = trials["choice"].isin([-1, 1])

    X = X[valid_mask]

    y = (
        trials.loc[
            valid_mask,
            "choice"
        ] == -1
    ).astype(int).values

    # --------------------------------------------------------
    # Basic quality checks


    if len(y) < MIN_TRIALS:
        return None, None

    if len(np.unique(y)) < 2:
        return None, None

    return X, y

In [4]:

# Logistic regression decoder


from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score,
)


DECODER = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        penalty="l2",
        C=0.1,
        solver="liblinear",
        max_iter=1000,
    ),
)

CV = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

In [ ]:

# Run baseline choice decoding

baseline_results = []

for region_group, region_list in BROAD_TARGET_REGIONS.items():

    for pid in all_pids:

        try:
            X, y = prepare_choice_decoder_data(
                pid=pid,
                region_acronyms=region_list,
                meta=meta,
            )

            if X is None:
                continue

            # Make sure each class has enough samples
            # for the requested number of CV folds.
            class_counts = np.bincount(y)

            if (
                len(class_counts) < 2
                or np.min(class_counts) < N_SPLITS
            ):
                continue

            scores = cross_val_score(
                DECODER,
                X,
                y,
                cv=CV,
                scoring="accuracy",
            )

            baseline_results.append({
                "PID": pid,
                "Region": region_group,
                "Accuracy": np.mean(scores),
            })

        except Exception as error:

            print(
                f"Skipping PID {pid} "
                f"({region_group}): {error}"
            )

In [ ]:

# Organize decoding results


df_baseline = pd.DataFrame(
    baseline_results
)

print(
    f"Number of successful probe insertions: "
    f"{df_baseline['PID'].nunique()}"
)

print(
    f"Number of decoding observations: "
    f"{len(df_baseline)}"
)

In [ ]:
baseline_summary = (
    df_baseline
    .groupby("Region")["Accuracy"]
    .agg(
        N="count",
        Mean="mean",
        SEM="sem",
        Median="median",
    )
    .reset_index()
)

print(
    baseline_summary.to_string(
        index=False
    )
)

In [ ]:

# Visualize baseline decoding

fig, ax = plt.subplots(
    figsize=(8, 5)
)

regions = baseline_summary["Region"]
means = baseline_summary["Mean"]
sems = baseline_summary["SEM"]

ax.errorbar(
    regions,
    means,
    yerr=sems,
    fmt="o",
    capsize=5,
)

ax.axhline(
    0.5,
    linestyle="--",
    linewidth=1.2,
)

ax.set_ylabel(
    "Choice decoding accuracy"
)

ax.set_xlabel(
    "Brain-region group"
)

ax.set_title(
    "Baseline Pre-Stimulus Choice Decoding"
)

ax.set_ylim(
    0.45,
    0.70,
)

ax.grid(
    axis="y",
    linestyle=":",
    alpha=0.5,
)

plt.tight_layout()
plt.show()

## 6. History-Dependent Choice Decoding

Choice behavior in the IBL task depends not only on the current
stimulus, but also on recent trial history.

To test whether neural representations of current choice are
history-dependent, we separate trials according to the outcome of
the previous trial:

- Previous trial rewarded
- Previous trial unrewarded

For each condition, current choice is decoded from neural activity
during the late pre-stimulus window (-0.4 to 0 s).

This analysis asks whether the neural representation of current
choice changes as a function of recent feedback history.

Importantly, this is still a choice-decoding analysis rather than
a direct test of subjective prior representation.

In [ ]:

# History-dependent choice decoding

HISTORY_WINDOW = (-0.4, 0.0)


def prepare_history_decoder_data(
    pid,
    region_acronyms,
    meta,
    time_window=HISTORY_WINDOW,
):
    """
    Prepare neural features and current-choice labels
    conditioned on previous-trial feedback.

    Returns
    -------
    X : np.ndarray or None
        Trial-by-unit neural activity matrix.

    y : np.ndarray or None
        Current-choice labels.

    prev_feedback : np.ndarray or None
        Previous-trial feedback labels.
    """

    clusters = meta.clusters[
        (meta.clusters["pid"] == pid)
        & (
            meta.clusters["acronym"]
            .isin(region_acronyms)
        )
    ]

    if len(clusters) < MIN_UNITS:
        return None, None, None

    uuids = clusters["uuids"].tolist()

    psth, _, trials = get_psth_for_clusters(
        uuids,
        meta,
    )

    if isinstance(psth, list):
        psth = psth[0]

    if isinstance(trials, list):
        trials = trials[0]

    if psth.shape[1] < MIN_UNITS:
        return None, None, None

    start_time, end_time = time_window

    time_mask = (
        (meta.times >= start_time)
        & (meta.times < end_time)
    )

    if not np.any(time_mask):
        return None, None, None

    X = psth[:, :, time_mask].mean(axis=2)

    # Previous-trial feedback
    prev_feedback = trials["feedbackType"].shift(1)

    # Valid current choice and previous feedback
    valid_mask = (
        trials["choice"].isin([-1, 1])
        & prev_feedback.isin([-1, 1])
    )

    X = X[valid_mask]

    y = (
        trials.loc[valid_mask, "choice"] == -1
    ).astype(int).values

    prev_feedback = (
        prev_feedback[valid_mask]
        .astype(int)
        .values
    )

    if len(y) < MIN_TRIALS:
        return None, None, None

    return X, y, prev_feedback

In [ ]:
HISTORY_DECODER = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        penalty="l2",
        C=0.1,
        solver="liblinear",
        max_iter=1000,
    ),
)

In [ ]:
HISTORY_CV = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

In [ ]:
history_results = []

for region_group, region_list in BROAD_TARGET_REGIONS.items():

    for pid in all_pids:

        try:

            X, y, prev_feedback = prepare_history_decoder_data(
                pid=pid,
                region_acronyms=region_list,
                meta=meta,
            )

            if X is None:
                continue

            for feedback_condition, feedback_value in [
                ("Previous reward", 1),
                ("Previous error", -1),
            ]:

                condition_mask = (
                    prev_feedback == feedback_value
                )

                X_condition = X[condition_mask]
                y_condition = y[condition_mask]

                if len(y_condition) < MIN_CONDITION_TRIALS:
                    continue

                # Require enough trials in both classes
                class_counts = np.bincount(y_condition)

                if (
                    len(class_counts) < 2
                    or np.min(class_counts) < N_SPLITS
                ):
                    continue

                scores = cross_val_score(
                    HISTORY_DECODER,
                    X_condition,
                    y_condition,
                    cv=HISTORY_CV,
                    scoring="accuracy",
                )

                history_results.append({
                    "PID": pid,
                    "Region": region_group,
                    "PreviousFeedback": feedback_condition,
                    "Accuracy": np.mean(scores),
                    "N_trials": len(y_condition),
                })

        except Exception as error:

            print(
                f"Skipping PID {pid} "
                f"({region_group}): {error}"
            )

In [ ]:
df_history = pd.DataFrame(
    history_results
)

print(
    f"Number of decoding observations: "
    f"{len(df_history)}"
)

print(
    f"Number of probe insertions: "
    f"{df_history['PID'].nunique()}"
)

df_history.head()

In [ ]:
history_summary = (
    df_history
    .groupby(
        ["Region", "PreviousFeedback"]
    )["Accuracy"]
    .agg(
        N="count",
        Mean="mean",
        SEM="sem",
        Median="median",
    )
    .reset_index()
)

print(
    history_summary.to_string(
        index=False
    )
)

In [ ]:
fig, ax = plt.subplots(
    figsize=(9, 5)
)

for feedback_condition in [
    "Previous reward",
    "Previous error",
]:

    subset = history_summary[
        history_summary["PreviousFeedback"]
        == feedback_condition
    ]

    ax.errorbar(
        subset["Region"],
        subset["Mean"],
        yerr=subset["SEM"],
        marker="o",
        capsize=5,
        label=feedback_condition,
    )

ax.axhline(
    0.5,
    linestyle="--",
    linewidth=1.2,
)

ax.set_xlabel(
    "Brain-region group"
)

ax.set_ylabel(
    "Choice decoding accuracy"
)

ax.set_title(
    "History-Dependent Pre-Stimulus Choice Decoding"
)

ax.legend()

ax.grid(
    axis="y",
    linestyle=":",
    alpha=0.5,
)

plt.tight_layout()
plt.show()

### Interpretation

Current-choice decoding was consistently higher following rewarded
trials than following unrewarded trials across all broad brain-region
groups.

This pattern suggests that recent feedback history is associated with
changes in the decodability of current choice from pre-stimulus neural
activity.

However, this analysis does not establish that feedback directly changes
neural choice representations. Differences may also reflect changes in
behavior, previous choice, latent task state, or prior beliefs.

Therefore, these results motivate explicitly modeling trial history and
subjective prior beliefs in the subsequent analyses.

## 7. Time-Resolved Choice Decoding

The previous analyses summarized neural activity over a fixed
pre-stimulus window (-0.4 to 0 s).

Here, we examine how choice decodability evolves over time.

For each probe insertion and brain-region group, neural activity is
averaged within short temporal bins and used to decode current choice.

The analysis covers the late pre-stimulus period and the early
post-stimulus period, allowing us to characterize when choice-related
information becomes decodable.

In [ ]:
TIME_RESOLVED_START = -0.40
TIME_RESOLVED_END = 0.80
TIME_BIN_WIDTH = 0.05

time_bins = np.arange(
    TIME_RESOLVED_START,
    TIME_RESOLVED_END + TIME_BIN_WIDTH,
    TIME_BIN_WIDTH,
)

bin_centers = (
    time_bins[:-1] + time_bins[1:]
) / 2

In [ ]:
def prepare_time_resolved_data(
    pid,
    region_acronyms,
    meta,
):
    """
    Prepare trial-by-unit neural activity for
    time-resolved choice decoding.
    """

    clusters = meta.clusters[
        (meta.clusters["pid"] == pid)
        & (
            meta.clusters["acronym"]
            .isin(region_acronyms)
        )
    ]

    if len(clusters) < MIN_UNITS:
        return None, None

    uuids = clusters["uuids"].tolist()

    psth, _, trials = get_psth_for_clusters(
        uuids,
        meta,
    )

    if isinstance(psth, list):
        psth = psth[0]

    if isinstance(trials, list):
        trials = trials[0]

    if psth.shape[1] < MIN_UNITS:
        return None, None

    valid_mask = trials["choice"].isin([-1, 1])

    psth = psth[valid_mask]

    y = (
        trials.loc[
            valid_mask,
            "choice"
        ] == -1
    ).astype(int).values

    if len(y) < MIN_TRIALS:
        return None, None

    if len(np.unique(y)) < 2:
        return None, None

    return psth, y

In [ ]:
time_resolved_results = []

for region_group, region_list in BROAD_TARGET_REGIONS.items():

    for pid in all_pids:

        try:

            psth, y = prepare_time_resolved_data(
                pid=pid,
                region_acronyms=region_list,
                meta=meta,
            )

            if psth is None:
                continue

            class_counts = np.bincount(y)

            if (
                len(class_counts) < 2
                or np.min(class_counts) < N_SPLITS
            ):
                continue

            for bin_idx in range(len(time_bins) - 1):

                start_time = time_bins[bin_idx]
                end_time = time_bins[bin_idx + 1]

                time_mask = (
                    (meta.times >= start_time)
                    & (meta.times < end_time)
                )

                if not np.any(time_mask):
                    continue

                X = psth[:, :, time_mask].mean(axis=2)

                scores = cross_val_score(
                    DECODER,
                    X,
                    y,
                    cv=CV,
                    scoring="accuracy",
                )

                time_resolved_results.append({
                    "PID": pid,
                    "Region": region_group,
                    "Time": (
                        start_time + end_time
                    ) / 2,
                    "Accuracy": np.mean(scores),
                })

        except Exception as error:

            print(
                f"Skipping PID {pid} "
                f"({region_group}): {error}"
            )

In [ ]:
df_time_resolved = pd.DataFrame(
    time_resolved_results
)

print(
    f"Number of decoding observations: "
    f"{len(df_time_resolved)}"
)

print(
    f"Number of probe insertions: "
    f"{df_time_resolved['PID'].nunique()}"
)

df_time_resolved.head()

In [ ]:
time_summary = (
    df_time_resolved
    .groupby(
        ["Region", "Time"]
    )["Accuracy"]
    .agg(
        N="count",
        Mean="mean",
        SEM="sem",
    )
    .reset_index()
)

In [ ]:
fig, ax = plt.subplots(
    figsize=(10, 6)
)

for region in BROAD_TARGET_REGIONS.keys():

    subset = time_summary[
        time_summary["Region"] == region
    ]

    ax.plot(
        subset["Time"],
        subset["Mean"],
        marker="o",
        markersize=3,
        label=region,
    )

    ax.fill_between(
        subset["Time"],
        subset["Mean"] - subset["SEM"],
        subset["Mean"] + subset["SEM"],
        alpha=0.15,
    )

ax.axhline(
    0.5,
    linestyle="--",
    linewidth=1.2,
)

ax.axvline(
    0,
    linestyle=":",
    linewidth=1.2,
)

ax.set_xlabel(
    "Time relative to stimulus onset (s)"
)

ax.set_ylabel(
    "Choice decoding accuracy"
)

ax.set_title(
    "Time-Resolved Pre- and Post-Stimulus Choice Decoding"
)

ax.legend()

ax.grid(
    axis="y",
    linestyle=":",
    alpha=0.5,
)

plt.tight_layout()
plt.show()

### Interpretation

Choice decoding showed relatively stable performance during the
pre-stimulus interval (-0.4 to 0 s), with no clear temporal increase
across the four broad region groups.

Because pre-stimulus neural activity can reflect multiple sources of
information, including previous choice, previous feedback, block state,
latent task state, and motor preparation, pre-stimulus choice decoding
cannot by itself be interpreted as a neural representation of prior
belief.

Following stimulus onset, decoding accuracy increased markedly in the
Motor and Integrative groups. This indicates that current choice became
more strongly decodable from population activity after stimulus onset.

These results motivate a more specific analysis of the information
available before stimulus onset, rather than interpreting pre-stimulus
choice decoding as evidence for prior representation.

## 8. Temporal Window × History Analysis

The previous analyses showed two descriptive patterns:

1. Current-choice decoding differed according to previous-trial feedback.
2. Choice decoding changed over time, particularly after stimulus onset.

Here, we examine whether the relationship between previous-trial
feedback and current-choice decoding varies across time.

For each time bin, current choice is decoded separately following
rewarded and unrewarded trials.

This analysis is descriptive and is intended to characterize the
temporal structure of history-dependent choice decoding.

A difference between feedback conditions is not interpreted as direct
evidence for changes in prior belief or a specific neural mechanism.

In [ ]:
def prepare_time_history_data(
    pid,
    region_acronyms,
    meta,
):
    """
    Prepare neural activity, current-choice labels,
    and previous-trial feedback for time-resolved
    history-dependent decoding.
    """

    clusters = meta.clusters[
        (meta.clusters["pid"] == pid)
        & (
            meta.clusters["acronym"]
            .isin(region_acronyms)
        )
    ]

    if len(clusters) < MIN_UNITS:
        return None, None, None

    uuids = clusters["uuids"].tolist()

    psth, _, trials = get_psth_for_clusters(
        uuids,
        meta,
    )

    if isinstance(psth, list):
        psth = psth[0]

    if isinstance(trials, list):
        trials = trials[0]

    if psth.shape[1] < MIN_UNITS:
        return None, None, None

    # Previous-trial feedback
    prev_feedback = trials["feedbackType"].shift(1)

    # Keep only trials with valid current choice
    # and valid previous feedback
    valid_mask = (
        trials["choice"].isin([-1, 1])
        & prev_feedback.isin([-1, 1])
    )

    psth = psth[valid_mask]

    y = (
        trials.loc[
            valid_mask,
            "choice"
        ] == -1
    ).astype(int).values

    prev_feedback = (
        prev_feedback[valid_mask]
        .astype(int)
        .values
    )

    if len(y) < MIN_TRIALS:
        return None, None, None

    return psth, y, prev_feedback

In [ ]:
time_history_results = []

for region_group, region_list in BROAD_TARGET_REGIONS.items():

    for pid in all_pids:

        try:

            psth, y, prev_feedback = (
                prepare_time_history_data(
                    pid=pid,
                    region_acronyms=region_list,
                    meta=meta,
                )
            )

            if psth is None:
                continue

            for bin_idx in range(len(time_bins) - 1):

                start_time = time_bins[bin_idx]
                end_time = time_bins[bin_idx + 1]

                time_mask = (
                    (meta.times >= start_time)
                    & (meta.times < end_time)
                )

                if not np.any(time_mask):
                    continue

                X = psth[:, :, time_mask].mean(axis=2)

                for feedback_condition, feedback_value in [
                    ("Previous reward", 1),
                    ("Previous error", -1),
                ]:

                    condition_mask = (
                        prev_feedback == feedback_value
                    )

                    X_condition = X[condition_mask]
                    y_condition = y[condition_mask]

                    if (
                        len(y_condition)
                        < MIN_CONDITION_TRIALS
                    ):
                        continue

                    class_counts = np.bincount(
                        y_condition
                    )

                    if (
                        len(class_counts) < 2
                        or np.min(class_counts) < N_SPLITS
                    ):
                        continue

                    scores = cross_val_score(
                        HISTORY_DECODER,
                        X_condition,
                        y_condition,
                        cv=HISTORY_CV,
                        scoring="accuracy",
                    )

                    time_history_results.append({
                        "PID": pid,
                        "Region": region_group,
                        "Time": (
                            start_time + end_time
                        ) / 2,
                        "PreviousFeedback": (
                            feedback_condition
                        ),
                        "Accuracy": np.mean(scores),
                        "N_trials": len(y_condition),
                    })

        except Exception as error:

            print(
                f"Skipping PID {pid} "
                f"({region_group}): {error}"
            )

In [ ]:
df_time_history = pd.DataFrame(
    time_history_results
)

print(
    f"Number of observations: "
    f"{len(df_time_history)}"
)

print(
    f"Number of probe insertions: "
    f"{df_time_history['PID'].nunique()}"
)

df_time_history.head()

In [ ]:
time_history_summary = (
    df_time_history
    .groupby(
        [
            "Region",
            "Time",
            "PreviousFeedback",
        ]
    )["Accuracy"]
    .agg(
        N="count",
        Mean="mean",
        SEM="sem",
        Median="median",
    )
    .reset_index()
)

In [ ]:
region = "Motor"


subset = time_history_summary[
    time_history_summary["Region"] == region
]

fig, ax = plt.subplots(
    figsize=(9, 5)
)

for feedback_condition in [
    "Previous reward",
    "Previous error",
]:

    condition = subset[
        subset["PreviousFeedback"]
        == feedback_condition
    ]

    ax.plot(
        condition["Time"],
        condition["Mean"],
        marker="o",
        markersize=3,
        label=feedback_condition,
    )

    ax.fill_between(
        condition["Time"],
        condition["Mean"] - condition["SEM"],
        condition["Mean"] + condition["SEM"],
        alpha=0.15,
    )

ax.axhline(
    0.5,
    linestyle="--",
    linewidth=1.2,
)

ax.axvline(
    0,
    linestyle=":",
    linewidth=1.2,
)

ax.set_xlabel(
    "Time relative to stimulus onset (s)"
)

ax.set_ylabel(
    "Choice decoding accuracy"
)

ax.set_title(
    f"Time-Resolved Choice Decoding — {region}"
)

ax.legend()

ax.grid(
    axis="y",
    linestyle=":",
    alpha=0.5,
)

plt.tight_layout()
plt.show()

In [ ]:
region = "Sensory"

subset = time_history_summary[
    time_history_summary["Region"] == region
]

fig, ax = plt.subplots(
    figsize=(9, 5)
)

for feedback_condition in [
    "Previous reward",
    "Previous error",
]:

    condition = subset[
        subset["PreviousFeedback"]
        == feedback_condition
    ]

    ax.plot(
        condition["Time"],
        condition["Mean"],
        marker="o",
        markersize=3,
        label=feedback_condition,
    )

    ax.fill_between(
        condition["Time"],
        condition["Mean"] - condition["SEM"],
        condition["Mean"] + condition["SEM"],
        alpha=0.15,
    )

ax.axhline(
    0.5,
    linestyle="--",
    linewidth=1.2,
)

ax.axvline(
    0,
    linestyle=":",
    linewidth=1.2,
)

ax.set_xlabel(
    "Time relative to stimulus onset (s)"
)

ax.set_ylabel(
    "Choice decoding accuracy"
)

ax.set_title(
    f"Time-Resolved Choice Decoding — {region}"
)

ax.legend()

ax.grid(
    axis="y",
    linestyle=":",
    alpha=0.5,
)

plt.tight_layout()
plt.show()

In [ ]:
region = "Memory"

subset = time_history_summary[
    time_history_summary["Region"] == region
]

fig, ax = plt.subplots(
    figsize=(9, 5)
)

for feedback_condition in [
    "Previous reward",
    "Previous error",
]:

    condition = subset[
        subset["PreviousFeedback"]
        == feedback_condition
    ]

    ax.plot(
        condition["Time"],
        condition["Mean"],
        marker="o",
        markersize=3,
        label=feedback_condition,
    )

    ax.fill_between(
        condition["Time"],
        condition["Mean"] - condition["SEM"],
        condition["Mean"] + condition["SEM"],
        alpha=0.15,
    )

ax.axhline(
    0.5,
    linestyle="--",
    linewidth=1.2,
)

ax.axvline(
    0,
    linestyle=":",
    linewidth=1.2,
)

ax.set_xlabel(
    "Time relative to stimulus onset (s)"
)

ax.set_ylabel(
    "Choice decoding accuracy"
)

ax.set_title(
    f"Time-Resolved Choice Decoding — {region}"
)

ax.legend()

ax.grid(
    axis="y",
    linestyle=":",
    alpha=0.5,
)

plt.tight_layout()
plt.show()

In [ ]:
region = "Integrative"

subset = time_history_summary[
    time_history_summary["Region"] == region
]

fig, ax = plt.subplots(
    figsize=(9, 5)
)

for feedback_condition in [
    "Previous reward",
    "Previous error",
]:

    condition = subset[
        subset["PreviousFeedback"]
        == feedback_condition
    ]

    ax.plot(
        condition["Time"],
        condition["Mean"],
        marker="o",
        markersize=3,
        label=feedback_condition,
    )

    ax.fill_between(
        condition["Time"],
        condition["Mean"] - condition["SEM"],
        condition["Mean"] + condition["SEM"],
        alpha=0.15,
    )

ax.axhline(
    0.5,
    linestyle="--",
    linewidth=1.2,
)

ax.axvline(
    0,
    linestyle=":",
    linewidth=1.2,
)

ax.set_xlabel(
    "Time relative to stimulus onset (s)"
)

ax.set_ylabel(
    "Choice decoding accuracy"
)

ax.set_title(
    f"Time-Resolved Choice Decoding — {region}"
)

ax.legend()

ax.grid(
    axis="y",
    linestyle=":",
    alpha=0.5,
)

plt.tight_layout()
plt.show()